# ERA5 Prediction Analysis

Interactive inspection of single test predictions on ERA5 surface temperature; it saves no
paper figure (the last section writes a presentation map of the empirical target
covariance). The backbone is always the **U-Cast** (`DhariwalUNet`); only the *predictive
method* (Det / Diag / spectral diagonal / LorD / SCORE) varies, and it is inferred from the
config that Hydra logged alongside each run.

Given a logged run directory (`logs/.../<timestamp>/`, containing `.hydra/config.yaml`
and `checkpoints/`), `analyze_run(...)` rebuilds the model, loads the checkpoint,
runs one test sample, and produces:

1. **target · prediction · squared-error · predictive-std**
2. **target · mean prediction · three generated samples**

**Original space.** The datamodule target is a *residual* (delta):
`y = value@(day0 + lead_time) − value@day0`. Every field shown here is mapped
back to the absolute field by adding the day-0 target channel from the input, and
(when the data was normalized) denormalized to physical units (Kelvin). See the
*Residual reconstruction* section.

## Setup

In [ ]:
import os, sys, math
from pathlib import Path
print(sys.executable, sys.version)

# --- Project-root bootstrap: put the repo root on sys.path so `import src...`
# works, and export PROJECT_ROOT for any Hydra ${oc.env:PROJECT_ROOT} lookups. ---
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".project-root").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("PROJECT_ROOT", str(PROJECT_ROOT))

import numpy as np
import torch
import hydra
from omegaconf import OmegaConf
from src.utils.legacy import upgrade_legacy_targets
import matplotlib.pyplot as plt

from src.methods.base import _as_list
from src.methods import PredictiveMethod, Deterministic
from src.models.gaussian_predictive import LightningPredictiveModel
from src.datamodules.datasets import WB_INPUT, WB_DYNAMIC, WB_TARGET

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("project root:", PROJECT_ROOT, "| device:", DEVICE)

## Choose a run

Point `RUN_DIR` at any logged run directory. It must contain `.hydra/config.yaml`
and a `checkpoints/` folder. Both single-stage runs and SCORE runs (two-stage fit) work —
for the latter the stage-2 checkpoint is picked automatically.

## Dataloader from config

The datamodule is rebuilt from the *run's* config (not the repo's current
`configs/`), so its spatial resolution, lead time and normalization always match
the checkpoint. `preload` / `num_workers` are overridden for interactive use.

In [ ]:
_DM_CACHE = {}

def load_run_config(run_dir):
    """Load the fully-composed Hydra config logged with a run."""
    cfg_path = Path(run_dir) / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(f"No logged config at {cfg_path}")
    return upgrade_legacy_targets(OmegaConf.load(cfg_path))

def build_datamodule(cfg, preload=False, num_workers=0, use_cache=True):
    """Instantiate the run's datamodule and set up the test split.

    `preload`/`num_workers` are overridden for interactive use (lazy per-sample
    reads, no worker processes). Cached by the data-shaping fields so repeated
    calls for the same config reuse one datamodule.
    """
    dm_cfg = cfg.datamodule
    key = (dm_cfg._target_, dm_cfg.get("data_path"), dm_cfg.get("downscaling_factor"),
           dm_cfg.get("last_t_steps"), dm_cfg.get("lead_time"),
           dm_cfg.get("normalize"), preload)
    if use_cache and key in _DM_CACHE:
        return _DM_CACHE[key]
    dm = hydra.utils.instantiate(dm_cfg, preload=preload, num_workers=num_workers)
    dm.setup("test")
    if use_cache:
        _DM_CACHE[key] = dm
    return dm

## Residual reconstruction to original space

`WeatherBench.__getitem__` returns the target as a delta,
`y = value@(day0 + lead_time) − value@day0`, and stacks the dynamic input
variables var-major over `last_t_steps` timesteps. So the target variable at
forecast **day 0** lives at one specific input channel; adding it back undoes the
residual, and denormalizing with the target statistics recovers Kelvin.

In [ ]:
def target_channel_index(cfg):
    """Input channel holding the target variable at forecast day 0.

    Matches the var-major stacking in `WeatherBench.__getitem__`: the target
    variable's position among the dynamic vars, times `last_t_steps`, plus the
    last-input-day offset.
    """
    last_t = int(cfg.datamodule.last_t_steps)
    var_pos = WB_DYNAMIC.index(WB_TARGET[0])
    return var_pos * last_t + (last_t - 1)

def target_stats(cfg):
    """(mean, std) of the target variable, for denormalizing to physical units."""
    stats = np.load(Path(cfg.datamodule.data_path) / "era5_statistics.npy")
    mean, std = np.split(stats, 2, axis=-1)
    i = WB_INPUT.index(WB_TARGET[0])
    return float(mean.squeeze()[i]), float(std.squeeze()[i])

def to_original_space(field, day0, cfg, denormalize=True):
    """Map a residual field back to the absolute target field.

    `field` is a residual (target / mean / samples), `day0` the target variable at
    forecast day 0 in the same units. Their sum is the absolute field; when the
    data was normalized, optionally denormalize to Kelvin. `day0` (H, W) is
    broadcast over any trailing sample axis of `field`.
    """
    d0 = day0
    while d0.ndim < field.ndim:
        d0 = d0[..., None]
    absolute = field + d0
    if bool(cfg.datamodule.get("normalize", True)) and denormalize:
        mean_t, std_t = target_stats(cfg)
        absolute = absolute * std_t + mean_t
    return absolute

## Build the model and load the checkpoint

Mirrors `src.evaluation.evaluate`: the U-Cast backbone and the predictive method
are rebuilt from config, then the checkpoint's weights (backbone only) are loaded
into them. SCORE runs (two-stage fit) use their stage-2 checkpoint.

In [ ]:
_MODEL_CACHE = {}

def find_checkpoint(run_dir):
    """Locate the deployed checkpoint in a logged run.

    Staged (whitened) runs deploy `checkpoints/stage2/*.ckpt`; plain runs keep a
    single `checkpoints/*.ckpt`. Returns the newest match.
    """
    ck = Path(run_dir) / "checkpoints"
    found = sorted(ck.glob("stage2/*.ckpt")) or sorted(ck.glob("*.ckpt"))
    if not found:
        raise FileNotFoundError(f"No .ckpt under {ck}")
    return str(found[-1])

def build_model_from_run(run_dir, device=DEVICE, datamodule=None,
                         preload=False, use_cache=True):
    """Rebuild the u-cast model for a logged run and load its checkpoint.

    Returns (model, predictive_method, datamodule, cfg, ckpt_path). The backbone
    is always the u-cast `DhariwalUNet`; the predictive method is inferred from
    the config.
    """
    if use_cache and str(run_dir) in _MODEL_CACHE:
        return _MODEL_CACHE[str(run_dir)]
    cfg = load_run_config(run_dir)
    dm = datamodule or build_datamodule(cfg, preload=preload)

    # Eval bandwidth(s): gamma = sqrt(gamma_eval * d), d = target event dimension.
    d = dm.get_total_dimension()
    gamma_eval = [math.sqrt(f * d) for f in _as_list(cfg.get("gamma_eval", [1.0]))]

    method = hydra.utils.instantiate(
        cfg.method, out_channels=dm.get_channels(),
        gamma=gamma_eval, gamma_eval=gamma_eval)
    backbone = hydra.utils.instantiate(
        cfg.backbone, in_channels=dm.get_input_channels(),
        out_channels=method.out_channels, group_widths=method.group_widths)

    ckpt = find_checkpoint(run_dir)
    model = LightningPredictiveModel.load_from_checkpoint(
        ckpt, backbone=backbone, predictive_method=method, map_location=device)
    model.eval().to(device)
    method.training = False  # eval mode -> gamma_eval / n_eval_samples

    out = (model, method, dm, cfg, ckpt)
    if use_cache:
        _MODEL_CACHE[str(run_dir)] = out
    return out

## Analysis + plots

`analyze_run(...)` runs one test sample and returns a dict of *preliminary data*
(all fields in original space, plus metadata) and, by default, renders the two
plots. Set `make_plots=False` to get just the arrays.

The predictive standard deviation is the method's **analytic** per-grid-point
marginal std — `method.get_marg_std(pred)`, the closed-form `sqrt(diag(Σ))` — so
it is exact for every method (the diagonal itself for Diag, the row norms of
`diag(d) + UUᵀ` for LorD, the Parseval mean for the circulant spectral diagonal, and
the stage-1 scale `D` for SCORE) rather than a noisy ensemble estimate. Ensemble `samples` are still drawn,
but only for the generated-member plot. A deterministic run has no spread, so its
std is zero and its samples are degenerate.

In [ ]:
def _plot_lon(lon):
    """Map ERA5 longitudes in [0, 360) to a signed [-180, 180) convention.

    This regional domain crosses the prime meridian (e.g. 347.5 .. 359.5, 0 .. 42),
    so the raw values are non-monotonic; the signed form is monotonic and places
    east/west correctly under pcolormesh.
    """
    return ((np.asarray(lon, dtype=float) + 180.0) % 360.0) - 180.0

def _show(ax, data, lon, lat, title, cmap, vmin, vmax, cbar_label):
    # pcolormesh on the true (lon, lat) grid so the domain is placed and oriented
    # geographically (latitude is ascending -> north ends up at the top).
    im = ax.pcolormesh(lon, lat, data, cmap=cmap, vmin=vmin, vmax=vmax,
                       shading="auto")
    ax.set_title(title)
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    cb = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if cbar_label:
        cb.set_label(cbar_label)
    return im

def plot_prediction_summary(data):
    """target · prediction (mean) · squared error · predictive std."""
    target, mean = data["target"], data["mean"]
    vmin = float(min(target.min(), mean.min()))
    vmax = float(max(target.max(), mean.max()))
    u = data["units"]
    lon, lat = _plot_lon(data["lon"]), data["lat"]
    fig, axs = plt.subplots(1, 4, figsize=(20, 4.4))
    _show(axs[0], target, lon, lat, "Target", "RdBu_r", vmin, vmax, u)
    _show(axs[1], mean, lon, lat, "Prediction (mean)", "RdBu_r", vmin, vmax, u)
    _show(axs[2], data["mse_map"], lon, lat, "Squared error", "magma", 0, None, f"{u}²")
    _show(axs[3], data["std_map"], lon, lat, "Predictive std", "viridis", 0, None, u)
    fig.suptitle(f"{data['method']} · {data['prediction_time']} · "
                 f"sample {data['sample_index']}", y=1.03)
    fig.tight_layout()
    return fig

def plot_samples(data, n_show=3):
    """target · mean prediction · n_show generated samples (shared color scale)."""
    target, mean, samples = data["target"], data["mean"], data["samples"]
    n_show = min(n_show, samples.shape[-1])
    show = samples[..., :n_show]
    vmin = float(min(target.min(), mean.min(), show.min()))
    vmax = float(max(target.max(), mean.max(), show.max()))
    u = data["units"]
    lon, lat = _plot_lon(data["lon"]), data["lat"]
    ncol = 1 + n_show
    fig, axs = plt.subplots(1, ncol, figsize=(5 * ncol, 4.4))
    _show(axs[0], target, lon, lat, "Target", "RdBu_r", vmin, vmax, u)
    #_show(axs[1], mean, lon, lat, "Mean prediction", "RdBu_r", vmin, vmax, u)
    for i in range(n_show):
        _show(axs[1 + i], show[..., i], lon, lat, f"Sample {i + 1}", "RdBu_r",
              vmin, vmax, u)
    fig.suptitle(f"{data['method']} · generated ensemble members", y=1.03)
    fig.tight_layout()
    return fig

In [ ]:
def analyze_run(run_dir, sample_index=0, n_samples=64, n_show_samples=3,
                device=DEVICE, denormalize=True, datamodule=None, make_plots=True):
    """Build the model for a logged run, run one test sample, and summarize it.

    Returns a dict with the fields in original space (`target`, `mean`,
    `samples` [H, W, M], `mse_map`, `std_map`, `day0_field`), metadata, and — when
    `make_plots` is True — the two figures (`fig_summary`, `fig_samples`).

    The predictive `std_map` is the method's *analytic* per-grid-point marginal
    std (`method.get_marg_std`, the closed-form sqrt(diag(Σ))), not an empirical
    ensemble estimate; `samples` are still drawn for the ensemble-member plot.
    """
    model, method, dm, cfg, ckpt = build_model_from_run(
        run_dir, device=device, datamodule=datamodule)

    x, y = dm.test_dataset[sample_index]
    x = x.unsqueeze(0).to(device)
    y = y.unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x)
        marg_std = method.get_marg_std(pred)  # (1, C, H, W); zeros for deterministic

    is_det = isinstance(method, Deterministic)
    mean = pred if is_det else pred[..., 0]
    samples = method.get_samples(pred, n_samples=n_samples)  # (1, C, H, W, M)

    ch = target_channel_index(cfg)
    day0 = x[0, ch].detach().cpu().numpy()  # (H, W)

    target = to_original_space(y[0, 0].cpu().numpy(), day0, cfg, denormalize)
    mean_abs = to_original_space(mean[0, 0].cpu().numpy(), day0, cfg, denormalize)
    samples_abs = to_original_space(
        samples[0, 0].cpu().numpy(), day0, cfg, denormalize)  # (H, W, M)

    mse_map = (mean_abs - target) ** 2

    # Analytic marginal std, denormalized to original units. A std is a scale, so
    # it only picks up the multiplicative target std (no additive day0/mean shift
    # that `to_original_space` applies to locations).
    normalized = bool(cfg.datamodule.get("normalize", True))
    std_scale = target_stats(cfg)[1] if (normalized and denormalize) else 1.0
    std_map = marg_std[0, 0].detach().cpu().numpy() * std_scale  # (H, W)

    (lat, lon), t = dm.test_dataset.get_grid()
    times = np.asarray(t.values)
    data = {
        "run_dir": str(run_dir),
        "method": cfg.method._target_.split(".")[-1],
        "loss": cfg.method.get("loss"),
        "ckpt": ckpt,
        "sample_index": sample_index,
        "prediction_time": str(times[sample_index])[:10],
        "units": "K" if (normalized and denormalize) else "norm.",
        "target": target,
        "mean": mean_abs,
        "samples": samples_abs,
        "mse_map": mse_map,
        "std_map": std_map,
        "day0_field": day0,
        "lat": np.asarray(lat),   # latitude cell centers, from the dataloader grid
        "lon": np.asarray(lon),   # longitude cell centers in [0, 360), from the grid
    }
    if is_det:
        print("Note: deterministic method -> predictive std and samples are degenerate.")
    if make_plots:
        data["fig_summary"] = plot_prediction_summary(data)
        data["fig_samples"] = plot_samples(data, n_show=n_show_samples)
    return data

# Example visualizations

In [ ]:
PRELOAD = False   # False = lazy per-sample reads (fast to open); True = whole split in RAM
DENORMALIZE = True  # map back to physical units (Kelvin) when the data was normalized
sample_index = 186

cfg = load_run_config(PROJECT_ROOT / "logs/train/runs/2026-07-17_17-31-39")
datamodule = build_datamodule(cfg, preload=PRELOAD)
print("grid    :", datamodule.test_dataset.get_dimensions(),
      "| test samples:", len(datamodule.test_dataset))

# Det

In [ ]:
RUN_DIR = PROJECT_ROOT / "logs/train/runs/2026-07-17_17-31-39"

In [ ]:
result = analyze_run(RUN_DIR, sample_index=sample_index, n_samples=100, n_show_samples=3,
                     denormalize=DENORMALIZE)

# The returned "preliminary data" (arrays + metadata); figures shown above.
print("method          :", result["method"], "| loss:", result["loss"])
print("prediction time :", result["prediction_time"], "| units:", result["units"])
print("checkpoint      :", result["ckpt"])
print("target shape    :", result["target"].shape,
      "| samples shape:", result["samples"].shape)
print("RMSE            : %.3f %s" % (np.sqrt(result["mse_map"].mean()), result["units"]))
print("mean pred. std  : %.3f %s" % (result["std_map"].mean(), result["units"]))
plt.show()

# Diag

# Spectral diagonal (`spectral_diag`)

In [ ]:
RUN_DIR = PROJECT_ROOT / "logs/train/runs/2026-07-10_14-29-18"

In [ ]:
result = analyze_run(RUN_DIR, sample_index=sample_index, n_samples=100, n_show_samples=3,
                     denormalize=DENORMALIZE)

# The returned "preliminary data" (arrays + metadata); figures shown above.
print("method          :", result["method"], "| loss:", result["loss"])
print("prediction time :", result["prediction_time"], "| units:", result["units"])
print("checkpoint      :", result["ckpt"])
print("target shape    :", result["target"].shape,
      "| samples shape:", result["samples"].shape)
print("RMSE            : %.3f %s" % (np.sqrt(result["mse_map"].mean()), result["units"]))
print("mean pred. std  : %.3f %s" % (result["std_map"].mean(), result["units"]))
plt.show()

In [ ]:
RUN_DIR = PROJECT_ROOT / "logs/train/runs/2026-07-10_14-31-35"

In [ ]:
result = analyze_run(RUN_DIR, sample_index=sample_index, n_samples=100, n_show_samples=3,
                     denormalize=DENORMALIZE)

# The returned "preliminary data" (arrays + metadata); figures shown above.
print("method          :", result["method"], "| loss:", result["loss"])
print("prediction time :", result["prediction_time"], "| units:", result["units"])
print("checkpoint      :", result["ckpt"])
print("target shape    :", result["target"].shape,
      "| samples shape:", result["samples"].shape)
print("RMSE            : %.3f %s" % (np.sqrt(result["mse_map"].mean()), result["units"]))
print("mean pred. std  : %.3f %s" % (result["std_map"].mean(), result["units"]))
plt.show()

# LorD (NLL)

In [ ]:
RUN_DIR = PROJECT_ROOT / "logs/train/runs/2026-07-17_17-32-25"

In [ ]:
result = analyze_run(RUN_DIR, sample_index=sample_index, n_samples=100, n_show_samples=3,
                     denormalize=DENORMALIZE)

# The returned "preliminary data" (arrays + metadata); figures shown above.
print("method          :", result["method"], "| loss:", result["loss"])
print("prediction time :", result["prediction_time"], "| units:", result["units"])
print("checkpoint      :", result["ckpt"])
print("target shape    :", result["target"].shape,
      "| samples shape:", result["samples"].shape)
print("RMSE            : %.3f %s" % (np.sqrt(result["mse_map"].mean()), result["units"]))
print("mean pred. std  : %.3f %s" % (result["std_map"].mean(), result["units"]))
plt.show()

# SCORE (kernel score)

In [ ]:
RUN_DIR = PROJECT_ROOT / "logs/train/runs/2026-07-17_17-30-11" # T=1

In [ ]:
result = analyze_run(RUN_DIR, sample_index=sample_index, n_samples=100, n_show_samples=3,
                     denormalize=DENORMALIZE)

# The returned "preliminary data" (arrays + metadata); figures shown above.
print("method          :", result["method"], "| loss:", result["loss"])
print("prediction time :", result["prediction_time"], "| units:", result["units"])
print("checkpoint      :", result["ckpt"])
print("target shape    :", result["target"].shape,
      "| samples shape:", result["samples"].shape)
print("RMSE            : %.3f %s" % (np.sqrt(result["mse_map"].mean()), result["units"]))
print("mean pred. std  : %.3f %s" % (result["std_map"].mean(), result["units"]))
plt.show()

## Analyze correlation matrix

In [ ]:
device = "cuda"
model, method, dm, cfg, ckpt = build_model_from_run(
    RUN_DIR, device=device, datamodule=datamodule)

x, y = dm.test_dataset[sample_index]
x = x.unsqueeze(0).to(device)
y = y.unsqueeze(0).to(device)
with torch.no_grad():
    pred = model(x)


In [ ]:
mu, diag, s = pred[..., 0], pred[..., 1], pred[..., 2]

In [ ]:
plt.imshow(s[0,0].cpu().log())
plt.colorbar()

In [ ]:
s.mean()

# Presentation map — empirical covariance with one grid point

A single map for an ERA5 overview slide (not a paper figure): the **empirical covariance of the forecast target** at
every grid point with one anchor point in central Europe (`ANCHOR_LATLON`, default 50 °N 10 °E),
in K².

* **The field is the task's own target**, not raw temperature: `WeatherBench.__getitem__` predicts
  the change `t2m(day0 + lead_time) − t2m(day0)`, with `last_t_steps = 2` and `lead_time = 3` from
  `configs/datamodule/era5.yaml`. Over the test split (2020–2022) that is the same 1092 cases the
  model is scored on. The covariance is computed in Kelvin; the model's standardized units are
  the same map divided by $\sigma_{\mathrm{t2m}}^2$ (the target is a difference, so the
  standardization's mean cancels and only its std rescales it).
* **Map styling**: PlateCarree on the true lat/lon
  grid (0.25°, 35–74.75 °N / 12.5 °W–42.25 °E), so the panel has that projection's ratio —
  latitude span over longitude span, a degree drawn square — with coastlines and the latitudes and
  longitudes labelled as text at the same fixed ticks. The canvas height is trimmed to the panel,
  so the colorbar is exactly as tall as the map.

Self-contained: it needs only the setup cell at the top and reads the zarr store directly, so
none of the run directories above have to exist. Written to `figures/presentation/`
(git-ignored).

In [ ]:
# Presentation map: empirical covariance of the ERA5 forecast target with one anchor grid point.
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker

if str(PROJECT_ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import plot_config as pc

PLOT = pc.apply_style()

from src.utils.paths import configured_path
ERA5_ZARR     = str(configured_path("era5_dir") / "era5.zarr")
TEST_YEARS    = slice("2020", "2022")   # WeatherBench.get_split("test")
LAST_T_STEPS  = 2                       # configs/datamodule/era5.yaml
LEAD_TIME     = 3                       # days
ANCHOR_LATLON = (50.0, 10.0)            # (deg N, deg E): central Germany

t2m = xr.open_zarr(ERA5_ZARR, consolidated=False)["2m_temperature"].sel(time=TEST_YEARS)
T = t2m.values.astype(np.float64)                          # (days, lat, lon), Kelvin
day0 = np.arange(LAST_T_STEPS - 1, T.shape[0] - LEAD_TIME)  # day 0 of every test case
Y = T[day0 + LEAD_TIME] - T[day0]                          # the target, as in WeatherBench.__getitem__
lat = t2m.latitude.values.astype(np.float64)
lon = ((t2m.longitude.values.astype(np.float64) + 180.0) % 360.0) - 180.0   # 347.5..42.25 -> -12.5..42.25
ia = int(np.abs(lat - ANCHOR_LATLON[0]).argmin())
ja = int(np.abs(lon - ANCHOR_LATLON[1]).argmin())

R = Y - Y.mean(axis=0)
cov = np.einsum("t,thw->hw", R[:, ia, ja], R) / (R.shape[0] - 1)   # K^2, unbiased
vmax = float(np.abs(cov).max())
print(f"{Y.shape[0]} test cases, grid {Y.shape[1]}x{Y.shape[2]}, "
      f"anchor ({lat[ia]:.2f} N, {lon[ja]:.2f} E)")
print(f"anchor variance {cov[ia, ja]:.2f} K^2, covariance range {cov.min():+.2f} .. {cov.max():+.2f} K^2")

# ---- the map: PlateCarree + lat/lon text labels ----
fig, ax = pc.subplots(1, 1, aspect=1.0, columns="single",
                      subplot_kw={"projection": ccrs.PlateCarree()})
pcm = ax.pcolormesh(lon, lat, cov, shading="nearest", cmap=pc.diverging_cmap(),
                    vmin=-vmax, vmax=vmax, rasterized=True, transform=ccrs.PlateCarree())
ax.plot(lon[ja], lat[ia], "kx", ms=1.6 * PLOT.marker_size, mew=1.0,
        transform=ccrs.PlateCarree(), zorder=5)
# 50m coastlines, thinned so a single-column panel does not read as mostly coastline.
ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=PLOT.thin_line_width * 0.6, alpha=0.8)
ax.add_feature(cfeature.BORDERS.with_scale("50m"), edgecolor=pc.MAP_COLORS["border"],
               lw=PLOT.thin_line_width * 0.4)
ax.set_extent([lon[0], lon[-1], float(lat.min()), float(lat.max())], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={"bottom": "x", "left": "y"}, x_inline=False, y_inline=False)
gl.xlines, gl.ylines = False, False
gl.xlocator = mticker.FixedLocator([-10, 10, 30])
gl.ylocator = mticker.FixedLocator([40, 50, 60, 70])
gl.xlabel_style = gl.ylabel_style = {"size": PLOT.tick_size}
cb = fig.colorbar(pcm, ax=ax, fraction=0.05, pad=0.02)
cb.set_label(r"Covariance [K$^2$]")
cb.ax.tick_params(labelsize=PLOT.tick_size)

# PlateCarree fixes the panel's ratio, so constrained layout would leave the rest of a taller
# canvas as blank space above and below the map (and the colorbar would span it). Trim the
# canvas by the measured slack until the panel fills its slot.
for _ in range(5):
    fig.canvas.draw()
    slack = (ax.get_position(original=True).height - ax.get_position().height) * fig.get_figheight()
    if slack < 0.005:
        break
    fig.set_figheight(fig.get_figheight() - slack)
pc.save(fig, pc.PRESENTATION_DIR / "era5_empirical_covariance.pdf")
plt.show()
